# Multi-Model Comparison: Interval Detection

## Overview

This notebook provides a systematic comparison across ten interval detection models:

| Model | Type | Features |
|-------|------|----------|
| **XGBoost** | Gradient Boosting (tree-based) | Full feature set |
| **FFNN** | Feed-Forward Neural Network | Full feature set |
| **BiLSTM** | Bidirectional LSTM | Full feature set |
| **BiLSTM (reduced)** | Bidirectional LSTM | Reduced feature set |
| **TCN** | Temporal Convolutional Network (dilated) | Full feature set |
| **TCN (reduced)** | Temporal Convolutional Network (dilated) | Reduced feature set |
| **CNN** | Convolutional Neural Network (improved) | Full feature set |
| **CNN (reduced)** | Convolutional Neural Network (improved) | Reduced feature set |
| **U-Net** | 1D U-Net (encoder-decoder with skip connections) | Full feature set |
| **U-Net (reduced)** | 1D U-Net (encoder-decoder with skip connections) | Reduced feature set |

All models were evaluated on the same set of athletic training sessions using a **leave-one-session-out** cross-validation strategy. Each session is held out as a test set while the remaining sessions are used for training.

### Evaluation Metrics

| Metric | Description |
|--------|-------------|
| **F_beta** | Weighted harmonic mean of precision and recall (primary metric) |
| **Precision** | Fraction of predicted intervals that are correct (fewer false positives = higher precision) |
| **Recall** | Fraction of true intervals that were detected (fewer missed intervals = higher recall) |
| **MAE (seconds)** | Mean absolute timing error between predicted and true interval boundaries |

In [ ]:
# ============================================================================
# MULTI-MODEL COMPARISON ANALYSIS
# ============================================================================

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

print("=" * 70)
print("MULTI-MODEL COMPARISON ANALYSIS")
print("=" * 70)

# ============================================================================
# 1. DEFINE AND LOAD ALL MODELS
# ============================================================================

MODELS = OrderedDict([
    ('XGBoost',          'results/xgboost_results.pkl'),
    ('FFNN',             'results/FFNN_results.pkl'),
    ('BiLSTM',           'results/bilstm_results.pkl'),
    ('BiLSTM (reduced)', 'results/bilstm_reduced_results.pkl'),
    ('TCN',              'results/tcn_dilated_results.pkl'),
    ('TCN (reduced)',    'results/tcn_dilated_reduced_results.pkl'),
    ('CNN',              'results/cnn_improved_results.pkl'),
    ('CNN (reduced)',    'results/cnn_improved_reduced_results.pkl'),
    ('U-Net',            'results/unet_1d_results.pkl'),
    ('U-Net (reduced)',  'results/unet_1d_reduced_results.pkl'),
])

MODEL_STYLES = {
    'XGBoost':          {'color': '#1f77b4', 'marker': 's'},
    'FFNN':             {'color': '#ff7f0e', 'marker': 'o'},
    'BiLSTM':           {'color': '#2ca02c', 'marker': '^'},
    'BiLSTM (reduced)': {'color': '#98df8a', 'marker': 'v'},
    'TCN':              {'color': '#d62728', 'marker': 'D'},
    'TCN (reduced)':    {'color': '#ff9896', 'marker': 'd'},
    'CNN':              {'color': '#9467bd', 'marker': 'P'},
    'CNN (reduced)':    {'color': '#c5b0d5', 'marker': 'X'},
    'U-Net':            {'color': '#8c564b', 'marker': 'h'},
    'U-Net (reduced)':  {'color': '#c49c94', 'marker': 'H'},
}

# Load all results
all_results = {}  # {model_name: {session_id: result_dict}}
print("\nLoading models:")
for name, path in MODELS.items():
    with open(path, 'rb') as f:
        package = pickle.load(f)
    results_list = package['results']
    all_results[name] = {res['session']: res for res in results_list}
    print(f"  {name}: {len(results_list)} sessions")

# Find common sessions across ALL models
all_session_sets = [set(d.keys()) for d in all_results.values()]
common_sessions = set.intersection(*all_session_sets)
print(f"\nCommon sessions across all {len(MODELS)} models: {len(common_sessions)}")

for name in MODELS:
    extra = set(all_results[name].keys()) - common_sessions
    if extra:
        print(f"  {name} extra sessions (excluded): {len(extra)}")

# ============================================================================
# 2. BUILD LONG-FORMAT COMPARISON DATAFRAME
# ============================================================================

rows = []
for session_id in sorted(common_sessions):
    ref_res = all_results['XGBoost'][session_id]
    df_session = ref_res['df']

    sport = ref_res.get('sport', 'unknown')
    if sport == 'unknown':
        if 'Bike' in ref_res['athlete'] or 'Zwift' in session_id:
            sport = 'biking'
        else:
            sport = 'rowing'

    for model_name in MODELS:
        res = all_results[model_name][session_id]
        rows.append({
            'session': session_id,
            'athlete': ref_res['athlete'],
            'sport': sport,
            'n_intervals': ref_res['n_true'],
            'duration_sec': len(df_session),
            'model': model_name,
            'f_beta': res['f_beta'],
            'precision': res['precision'],
            'recall': res['recall'],
            'mae': res.get('mean_error_sec', np.nan),
            'n_pred': res['n_pred'],
            'n_true': res['n_true'],
        })

df_long = pd.DataFrame(rows)
df_long['complexity'] = pd.cut(
    df_long['n_intervals'],
    bins=[0, 10, 20, 100],
    labels=['Low (<=10)', 'Medium (11-20)', 'High (>20)']
)

print(f"\nDataframe: {len(df_long)} rows "
      f"({len(common_sessions)} sessions x {len(MODELS)} models)")

# ============================================================================
# 3. OVERALL PERFORMANCE RANKING
# ============================================================================

print("\n" + "=" * 70)
print("OVERALL PERFORMANCE RANKING (sorted by mean F_beta)")
print("=" * 70)

overall_stats = df_long.groupby('model').agg(
    F_beta_mean=('f_beta', 'mean'),
    F_beta_std=('f_beta', 'std'),
    Precision_mean=('precision', 'mean'),
    Recall_mean=('recall', 'mean'),
    MAE_mean=('mae', 'mean'),
).round(3).sort_values('F_beta_mean', ascending=False)

overall_stats.index.name = 'Model'
model_order = overall_stats.index.tolist()
print()
display(overall_stats)

# ============================================================================
# 4. BEST MODEL PER SESSION
# ============================================================================

print("\n" + "=" * 70)
print("BEST MODEL PER SESSION (by F_beta)")
print("=" * 70)

best_per_session = df_long.loc[df_long.groupby('session')['f_beta'].idxmax()]
win_counts = best_per_session['model'].value_counts()

print("\nSessions where each model achieved highest F_beta:")
for model, count in win_counts.items():
    pct = 100 * count / len(common_sessions)
    print(f"  {model}: {count} ({pct:.1f}%)")

# ============================================================================
# 5. BREAKDOWN BY SPORT
# ============================================================================

print("\n" + "=" * 70)
print("MEAN F_BETA BY SPORT")
print("=" * 70)

sport_table = df_long.pivot_table(
    values='f_beta', index='model', columns='sport', aggfunc='mean'
).round(3).loc[model_order]
print()
display(sport_table)

# ============================================================================
# 6. BREAKDOWN BY COMPLEXITY
# ============================================================================

print("\n" + "=" * 70)
print("MEAN F_BETA BY COMPLEXITY")
print("=" * 70)

comp_table = df_long.pivot_table(
    values='f_beta', index='model', columns='complexity',
    aggfunc='mean', observed=False
).round(3).loc[model_order]
print()
display(comp_table)

# ============================================================================
# 7. SESSION-BY-SESSION: BEST AND WORST MODELS
# ============================================================================

print("\n" + "=" * 70)
print("SESSION-BY-SESSION: BEST AND WORST MODELS")
print("=" * 70)

fbeta_wide = df_long.pivot_table(values='f_beta', index='session', columns='model')

print(f"\n{'Session':<48} {'Best Model':<20} {'Score':>6}  {'Worst Model':<20} {'Score':>6}")
print("-" * 110)
for session_id in sorted(common_sessions):
    row = fbeta_wide.loc[session_id]
    best_m = row.idxmax()
    worst_m = row.idxmin()
    s = session_id[:45] + "..." if len(session_id) > 48 else session_id
    print(f"  {s:<48} {best_m:<20} {row.max():>6.3f}  {worst_m:<20} {row.min():>6.3f}")

print("\n" + "=" * 70)
print("DATA LOADING COMPLETE")
print("=" * 70)

## Interpretation: Overall Comparison

### What to look for in the results above

- **Overall Ranking**: Models are sorted by mean F_beta. The top-ranked model achieves the best average detection quality across all sessions. However, average performance alone doesn't tell the whole story -- variance matters too.

- **Best Model Per Session**: Shows how often each model "wins" on individual sessions. A model with fewer overall wins may still be the best choice for specific session types.

- **By Sport**: Performance may differ between biking and rowing due to differences in heart rate dynamics. Some model architectures may capture sport-specific patterns better than others.

- **By Complexity**: Sessions with fewer intervals (Low) tend to be easier. Models that maintain performance on high-complexity sessions (many intervals) are more robust.

- **Full vs Reduced Features**: Models with "(reduced)" use a smaller feature set. If a reduced model performs comparably to its full-feature counterpart, the extra features add noise rather than signal.

- **Session-by-Session Best/Worst**: Highlights which sessions are easy (all models do well) and which are challenging (all models struggle). Large gaps between best and worst model on a session indicate that model choice matters significantly for that session type.

## Detailed Summary Tables

In [2]:
# ============================================================================
# SUMMARY TABLES
# ============================================================================

metrics = ['f_beta', 'precision', 'recall', 'mae']
metric_labels = ['F_beta', 'Precision', 'Recall', 'MAE (sec)']

# ── Table 1: All metrics for all models ──
print("=" * 60)
print("TABLE 1: Overall Metrics per Model")
print("=" * 60)

table1_rows = []
for model in model_order:
    mdf = df_long[df_long['model'] == model]
    table1_rows.append({
        'Model': model,
        'F_beta': mdf['f_beta'].mean(),
        'Precision': mdf['precision'].mean(),
        'Recall': mdf['recall'].mean(),
        'MAE (sec)': mdf['mae'].mean(),
    })
table1 = pd.DataFrame(table1_rows).set_index('Model').round(3)
display(table1)

# ── Table 2: F_beta per model per sport ──
print("\n" + "=" * 60)
print("TABLE 2: Metrics per Model per Sport")
print("=" * 60)

for metric, label in zip(metrics, metric_labels):
    print(f"\n--- {label} ---")
    pivot = df_long.pivot_table(
        values=metric, index='model', columns='sport', aggfunc='mean'
    ).round(3).loc[model_order]
    display(pivot)

# ── Table 3: F_beta per model per complexity ──
print("\n" + "=" * 60)
print("TABLE 3: Metrics per Model per Complexity")
print("=" * 60)

for metric, label in zip(metrics, metric_labels):
    print(f"\n--- {label} ---")
    pivot = df_long.pivot_table(
        values=metric, index='model', columns='complexity',
        aggfunc='mean', observed=False
    ).round(3).loc[model_order]
    display(pivot)

# ── Table 4: Win counts by sport and complexity ──
print("\n" + "=" * 60)
print("TABLE 4: Win Counts (sessions where model is best)")
print("=" * 60)

best = df_long.loc[df_long.groupby('session')['f_beta'].idxmax()]

print("\nBy Sport:")
win_sport = best.pivot_table(
    index='model', columns='sport', values='session',
    aggfunc='count', fill_value=0
)
win_sport['Total'] = win_sport.sum(axis=1)
win_sport = win_sport.sort_values('Total', ascending=False)
display(win_sport)

print("\nBy Complexity:")
win_comp = best.pivot_table(
    index='model', columns='complexity', values='session',
    aggfunc='count', fill_value=0, observed=False
)
win_comp['Total'] = win_comp.sum(axis=1)
win_comp = win_comp.sort_values('Total', ascending=False)
display(win_comp)

TABLE 1: Overall Metrics per Model


,F_beta,Precision,Recall,MAE (sec)
Model,,,,
CNN,0.735,0.733,0.736,24.524
FFNN,0.714,0.715,0.713,18.340
XGBoost,0.708,0.711,0.707,17.540
BiLSTM,0.692,0.691,0.692,26.783
CNN (reduced),0.602,0.600,0.602,65.054
TCN (reduced),0.562,0.561,0.563,72.454
BiLSTM (reduced),0.473,0.471,0.474,53.417
TCN,0.125,0.124,0.126,137.647



TABLE 2: Metrics per Model per Sport

--- F_beta ---


sport,biking,rowing
model,,
CNN,0.675,0.764
FFNN,0.661,0.740
XGBoost,0.594,0.765
BiLSTM,0.590,0.742
CNN (reduced),0.626,0.590
TCN (reduced),0.527,0.580
BiLSTM (reduced),0.444,0.487
TCN,0.099,0.138



--- Precision ---


...

complexity,Low (<=10),Medium (11-20),High (>20)
model,,,
CNN,29.334,22.843,17.426
FFNN,23.687,10.964,18.709
XGBoost,14.677,21.864,16.779
BiLSTM,21.040,25.350,40.417
CNN (reduced),92.771,36.641,52.237
TCN (reduced),94.936,32.075,88.058
BiLSTM (reduced),75.487,33.340,39.393
TCN,117.699,161.518,141.738



TABLE 4: Win Counts (sessions where model is best)

By Sport:


sport,biking,rowing,Total
model,,,
FFNN,3,3,6
CNN,2,3,5
XGBoost,1,4,5
BiLSTM,0,2,2



By Complexity:


complexity,Low (<=10),Medium (11-20),High (>20),Total
model,,,,
FFNN,2,3,1,6
CNN,1,2,2,5
XGBoost,4,1,0,5
BiLSTM,1,0,1,2


In [ ]:
# ============================================================================
# DETAILED METRICS
# ============================================================================

# ── 1. Model Consistency ──
print("=" * 60)
print("1. MODEL CONSISTENCY (F_beta Standard Deviation)")
print("=" * 60)

consistency = df_long.groupby('model')['f_beta'].std().round(3).sort_values()
consistency.name = 'F_beta Std Dev'
display(pd.DataFrame(consistency))
print(f"\nMost consistent:  {consistency.idxmin()} (std = {consistency.min():.3f})")
print(f"Least consistent: {consistency.idxmax()} (std = {consistency.max():.3f})")

# ── 2. Failure Rate ──
print("\n" + "=" * 60)
print("2. FAILURE RATE (sessions with F_beta < 0.5)")
print("=" * 60)

n_sessions = len(common_sessions)
failures = df_long.groupby('model')['f_beta'].apply(
    lambda g: (g < 0.5).sum()
).sort_values()
failure_pct = (failures / n_sessions * 100).round(1)
failure_df = pd.DataFrame({
    'Failures': failures,
    f'Rate (% of {n_sessions})': failure_pct
})
display(failure_df)

# ── 3. False Positive & False Negative Rates ──
print("\n" + "=" * 60)
print("3. FALSE POSITIVE & FALSE NEGATIVE RATES")
print("=" * 60)

fp_fn_rows = []
for model_name in model_order:
    model_df = df_long[df_long['model'] == model_name]
    fp_rates = []
    fn_rates = []
    for _, row in model_df.iterrows():
        tp = row['recall'] * row['n_true']
        fp = row['n_pred'] - tp
        fn = row['n_true'] - tp
        fp_rate = fp / row['n_pred'] if row['n_pred'] > 0 else 0
        fn_rate = fn / row['n_true'] if row['n_true'] > 0 else 0
        fp_rates.append(fp_rate)
        fn_rates.append(fn_rate)
    fp_fn_rows.append({
        'Model': model_name,
        'Avg FP Rate': np.mean(fp_rates),
        'Avg FN Rate': np.mean(fn_rates),
    })

fp_fn_df = pd.DataFrame(fp_fn_rows).set_index('Model').round(3)
display(fp_fn_df)

# ── 4. Full vs Reduced Feature Comparison ──
print("\n" + "=" * 60)
print("4. FULL vs REDUCED FEATURE SET COMPARISON")
print("=" * 60)

pairs = [
    ('BiLSTM', 'BiLSTM (reduced)'),
    ('TCN', 'TCN (reduced)'),
    ('CNN', 'CNN (reduced)'),
    ('U-Net', 'U-Net (reduced)'),
]

pair_rows = []
for full_name, reduced_name in pairs:
    full_scores = df_long[df_long['model'] == full_name].set_index('session')['f_beta']
    red_scores = df_long[df_long['model'] == reduced_name].set_index('session')['f_beta']

    full_mean = full_scores.mean()
    red_mean = red_scores.mean()
    diff = red_mean - full_mean

    full_wins = (full_scores > red_scores).sum()
    red_wins = (red_scores > full_scores).sum()
    ties = (full_scores == red_scores).sum()

    base = full_name.split(' ')[0]
    direction = "better" if diff > 0 else "worse" if diff < 0 else "same"

    pair_rows.append({
        'Architecture': base,
        'Full F_beta': round(full_mean, 3),
        'Reduced F_beta': round(red_mean, 3),
        'Diff (Red-Full)': round(diff, 3),
        'Full Wins': full_wins,
        'Reduced Wins': red_wins,
        'Ties': ties,
    })

    print(f"\n{base}:")
    print(f"  Full features:    {full_mean:.3f}")
    print(f"  Reduced features: {red_mean:.3f}")
    print(f"  Difference:       {diff:+.3f} (reduced is {direction})")
    print(f"  Full wins: {full_wins}, Reduced wins: {red_wins}, Ties: {ties}")

print()
display(pd.DataFrame(pair_rows).set_index('Architecture'))

## Interpretation: Detailed Metrics

### Model Consistency

The standard deviation of F_beta across sessions indicates how predictable a model's performance is. A model with lower standard deviation is more consistent -- the user can expect similar quality regardless of the session. A model with high variability might score excellently on some sessions but poorly on others.

### Failure Rate

Sessions with F_beta < 0.5 represent cases where the model's predictions are more wrong than right. A lower failure rate indicates a model that is more reliable across diverse session types. Even a model with high average F_beta can be problematic if it frequently fails on certain sessions.

### False Positive and False Negative Rates

- **False Positives** (predicting an interval that doesn't exist): Increase the manual review burden. A high FP rate means the model is "over-detecting."
- **False Negatives** (missing a real interval): Generally more costly, as missed intervals represent lost training data.

### Full vs Reduced Feature Comparison

This analysis reveals whether reducing the feature set helps or hurts each architecture:
- If the reduced model performs **comparably or better**, the full feature set contains redundant or noisy features for that architecture.
- If the reduced model performs **worse**, the extra features provide useful information that the model can leverage.
- Different architectures may respond differently to feature reduction, revealing which models are more robust to input dimensionality.

In [ ]:
# ============================================================================
# VISUALIZATIONS
# ============================================================================

import os
SAVE_DIR = 'results/multimodel_comparison'
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 70)
print(f"GENERATING VISUALIZATIONS (saving to {SAVE_DIR}/)")
print("=" * 70)

# ── Plot 1: Overall F_beta bar chart with error bars ──
fig, ax = plt.subplots(figsize=(14, 6))
means = [overall_stats.loc[m, 'F_beta_mean'] for m in model_order]
stds = [overall_stats.loc[m, 'F_beta_std'] for m in model_order]
colors = [MODEL_STYLES[m]['color'] for m in model_order]

bars = ax.bar(model_order, means, yerr=stds, capsize=5, color=colors,
              edgecolor='black', linewidth=0.5, alpha=0.85)
ax.set_ylabel('Mean F_beta', fontsize=13)
ax.set_title('Overall Model Performance (Mean F_beta +/- Std Dev)', fontsize=15)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/overall_fbeta.png', dpi=300, bbox_inches='tight')
print(f"  Saved: {SAVE_DIR}/overall_fbeta.png")
plt.show()

# ── Plot 2: Box plots of F_beta distribution ──
fig, ax = plt.subplots(figsize=(16, 7))
data = [df_long[df_long['model'] == m]['f_beta'].values for m in model_order]
bp = ax.boxplot(data, labels=model_order, patch_artist=True,
                showmeans=True, meanline=True)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax.set_ylabel('F_beta', fontsize=13)
ax.set_title('F_beta Distribution Across Sessions', fontsize=15)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fbeta_boxplot.png', dpi=300, bbox_inches='tight')
print(f"  Saved: {SAVE_DIR}/fbeta_boxplot.png")
plt.show()

# ── Plot 3: Grouped bar chart by sport ──
fig, ax = plt.subplots(figsize=(16, 7))
sports = sorted(df_long['sport'].unique())
x = np.arange(len(sports))
width = 0.8 / len(model_order)

for i, model in enumerate(model_order):
    vals = [df_long[(df_long['model'] == model) & (df_long['sport'] == s)]['f_beta'].mean()
            for s in sports]
    ax.bar(x + i * width, vals, width, label=model,
           color=MODEL_STYLES[model]['color'], edgecolor='black', linewidth=0.3)

ax.set_xlabel('Sport', fontsize=13)
ax.set_ylabel('Mean F_beta', fontsize=13)
ax.set_title('Model Performance by Sport', fontsize=15)
ax.set_xticks(x + width * len(model_order) / 2)
ax.set_xticklabels(sports)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/by_sport.png', dpi=300, bbox_inches='tight')
print(f"  Saved: {SAVE_DIR}/by_sport.png")
plt.show()

# ── Plot 4: Grouped bar chart by complexity ──
fig, ax = plt.subplots(figsize=(16, 7))
comp_labels = ['Low (<=10)', 'Medium (11-20)', 'High (>20)']
x = np.arange(len(comp_labels))
width = 0.8 / len(model_order)

for i, model in enumerate(model_order):
    vals = [df_long[(df_long['model'] == model) & (df_long['complexity'] == c)]['f_beta'].mean()
            for c in comp_labels]
    ax.bar(x + i * width, vals, width, label=model,
           color=MODEL_STYLES[model]['color'], edgecolor='black', linewidth=0.3)

ax.set_xlabel('Session Complexity', fontsize=13)
ax.set_ylabel('Mean F_beta', fontsize=13)
ax.set_title('Model Performance by Session Complexity', fontsize=15)
ax.set_xticks(x + width * len(model_order) / 2)
ax.set_xticklabels(comp_labels)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/by_complexity.png', dpi=300, bbox_inches='tight')
print(f"  Saved: {SAVE_DIR}/by_complexity.png")
plt.show()

# ── Plot 5: Heatmap - F_beta per model per session ──
fig, ax = plt.subplots(figsize=(20, max(8, len(common_sessions) * 0.5)))
heatmap_data = fbeta_wide[model_order].T
short_labels = [s[:28] + '..' if len(s) > 30 else s for s in heatmap_data.columns]
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0, vmax=1, ax=ax, linewidths=0.5,
            xticklabels=short_labels)
ax.set_title('F_beta by Model and Session', fontsize=15)
ax.set_ylabel('Model', fontsize=13)
ax.set_xlabel('Session', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/heatmap.png', dpi=300, bbox_inches='tight')
print(f"  Saved: {SAVE_DIR}/heatmap.png")
plt.show()

# ── Plot 6: Radar chart ──
radar_metrics = ['F_beta_mean', 'Precision_mean', 'Recall_mean']
radar_labels = ['F_beta', 'Precision', 'Recall']

# Add MAE inverted (lower MAE = better, so invert for radar)
max_mae = overall_stats['MAE_mean'].max()
if max_mae > 0:
    overall_stats['MAE_inv'] = 1 - (overall_stats['MAE_mean'] / (max_mae * 1.1))
else:
    overall_stats['MAE_inv'] = 1.0
radar_metrics.append('MAE_inv')
radar_labels.append('Timing Accuracy\n(1 - norm. MAE)')

# Add consistency (lower std = better)
max_std = overall_stats['F_beta_std'].max()
if max_std > 0:
    overall_stats['Consistency'] = 1 - (overall_stats['F_beta_std'] / (max_std * 1.1))
else:
    overall_stats['Consistency'] = 1.0
radar_metrics.append('Consistency')
radar_labels.append('Consistency\n(1 - norm. StdDev)')

n_radar = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, n_radar, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
for model in model_order:
    values = [overall_stats.loc[model, m] for m in radar_metrics]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2,
            color=MODEL_STYLES[model]['color'], label=model, markersize=5)
    ax.fill(angles, values, alpha=0.03, color=MODEL_STYLES[model]['color'])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Multi-Metric Model Comparison', fontsize=15, pad=20)
ax.legend(bbox_to_anchor=(1.35, 1.1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/radar.png', dpi=300, bbox_inches='tight')
print(f"  Saved: {SAVE_DIR}/radar.png")
plt.show()

# ── Plot 7: Full vs Reduced comparison ──
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
pairs = [('BiLSTM', 'BiLSTM (reduced)'),
         ('TCN', 'TCN (reduced)'),
         ('CNN', 'CNN (reduced)'),
         ('U-Net', 'U-Net (reduced)')]

for ax, (full_name, red_name) in zip(axes.flat, pairs):
    full_scores = df_long[df_long['model'] == full_name].set_index('session')['f_beta']
    red_scores = df_long[df_long['model'] == red_name].set_index('session')['f_beta']

    ax.scatter(full_scores, red_scores, s=80, alpha=0.7,
               color=MODEL_STYLES[full_name]['color'], edgecolors='black')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Equal performance')
    ax.set_xlabel(f'{full_name} F_beta', fontsize=11)
    ax.set_ylabel(f'{red_name} F_beta', fontsize=11)
    ax.set_title(f'{full_name.split(" ")[0]}: Full vs Reduced', fontsize=13)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/full_vs_reduced.png', dpi=300, bbox_inches='tight')
print(f"  Saved: {SAVE_DIR}/full_vs_reduced.png")
plt.show()

# ── Save comparison data ──
df_long.to_csv(f'{SAVE_DIR}/multimodel_comparison.csv', index=False)
print(f"\nSaved: {SAVE_DIR}/multimodel_comparison.csv")

print("\n" + "=" * 70)
print("VISUALIZATION COMPLETE")
print("=" * 70)

## Interpretation: Visualizations

### Overall Bar Chart (Plot 1)
Models are ranked by mean F_beta with error bars showing standard deviation. Taller bars indicate better average performance, while shorter error bars indicate more consistent performance. The ideal model has a tall bar with a short error bar.

### Box Plots (Plot 2)
Shows the full distribution of F_beta across sessions for each model. The box spans the interquartile range (IQR), the line inside is the median, and the dashed line is the mean. Outliers (dots beyond the whiskers) represent unusually good or poor sessions. A model with a higher median and tighter box is preferred.

### By Sport (Plot 3)
Reveals which models are strongest for biking vs rowing. Some architectures may capture sport-specific heart rate patterns better than others. This helps in selecting the best model when the sport type is known.

### By Complexity (Plot 4)
Shows how models handle different numbers of intervals. Models that maintain performance at high complexity are more robust. A steep drop-off from Low to High complexity indicates the model struggles with many closely-spaced intervals.

### Heatmap (Plot 5)
Provides a complete view of every model's performance on every session. Green cells indicate high F_beta, red cells indicate low F_beta. Columns (sessions) that are consistently green are "easy" sessions; columns with mixed colors are "hard" sessions where model choice matters most.

### Radar Chart (Plot 6)
Compares models across five dimensions simultaneously: F_beta, Precision, Recall, Timing Accuracy (inverted MAE), and Consistency (inverted std dev). Models with larger areas cover more dimensions well. This chart makes it easy to identify models that excel in specific aspects vs. models that are well-rounded.

### Full vs Reduced (Plot 7)
Scatter plots comparing each architecture's full and reduced feature variants session-by-session. Points above the diagonal mean the reduced model performed better on that session; points below mean the full model was better. Clustering near the diagonal indicates the feature reduction had little impact.

# Results Interpretation

## Overall Ranking

The eight models fall into **three clear performance tiers**:

| Tier | Models | F_beta Range | Description |
|------|--------|-------------|-------------|
| **Tier 1 (Competitive)** | CNN (0.735), FFNN (0.714), XGBoost (0.708), BiLSTM (0.692) | 0.69 -- 0.74 | Viable candidates for deployment |
| **Tier 2 (Moderate)** | CNN reduced (0.602), TCN reduced (0.562) | 0.56 -- 0.60 | Functional but significantly weaker |
| **Tier 3 (Poor)** | BiLSTM reduced (0.473), TCN full (0.125) | < 0.50 | Not usable in practice |

## Best Overall Model: CNN

**CNN achieves the highest mean F_beta (0.735)**, but there is no single dominant model across all conditions. The top four Tier 1 models each have distinct strengths:

- **CNN**: Highest average F_beta (0.735). Most robust to session complexity -- it degrades the least from low to high interval counts (only -0.056 drop). Best model for **biking** sessions (0.675). Lowest false positive and false negative rates (0.264 each).
- **FFNN**: Wins the **most individual sessions** (6 out of 18 = 33%). Strong balance of detection quality (F_beta = 0.714) with good timing accuracy (MAE = 18.3s). Best model for **medium-complexity** sessions (0.759). Tied for lowest failure rate (5.6%).
- **XGBoost**: **Best timing accuracy** of all models (MAE = 17.5s vs CNN's 24.5s). Strongest on **rowing** sessions (0.765) and dominates **low-complexity** sessions (wins 4 of 8). However, performance drops steeply on high-complexity sessions (-0.114 from low to high).
- **BiLSTM**: Tied with FFNN for the **lowest failure rate** (1 out of 18 = 5.6%). Most consistent among competitive models (std = 0.153). Rarely fails catastrophically, but also rarely wins (only 2/18 session wins).

## Key Finding: TCN (Full Features) is Broken

The TCN model with full features has a **100% failure rate** -- every single session scores below F_beta = 0.5, with a mean of just 0.125. This is far below what would be expected from the architecture and strongly suggests an issue with training (e.g., overfitting, convergence failure, or a hyperparameter problem). Notably, **TCN with reduced features works dramatically better** (0.562), which reinforces the hypothesis that the full-feature TCN cannot handle the input dimensionality.

## Full vs Reduced Feature Sets

The impact of feature reduction varies drastically across architectures:

| Architecture | Full F_beta | Reduced F_beta | Effect | Session Wins (Full / Reduced) |
|-------------|------------|---------------|--------|-------------------------------|
| **BiLSTM** | 0.692 | 0.473 | Full much better (-0.219) | 15 / 1 |
| **CNN** | 0.735 | 0.602 | Full much better (-0.133) | 12 / 3 |
| **TCN** | 0.125 | 0.562 | Reduced much better (+0.437) | 0 / 17 |

- **BiLSTM and CNN** clearly benefit from the full feature set -- removing features costs ~0.13--0.22 in F_beta.
- **TCN** is the exception: the full model is broken, and the reduced version recovers to moderate performance. This suggests TCN is highly sensitive to input dimensionality and may require more careful feature engineering or regularization.

## Sport-Specific Performance

**Biking** (6 sessions): CNN (0.675) > FFNN (0.661) > CNN reduced (0.626) > XGBoost (0.594) > BiLSTM (0.590)

**Rowing** (12 sessions): XGBoost (0.765) > CNN (0.764) > BiLSTM (0.742) > FFNN (0.740)

- For **rowing**, XGBoost and CNN are essentially tied (0.765 vs 0.764), with XGBoost having much better timing accuracy (MAE = 15.5s vs 24.9s).
- For **biking**, CNN leads clearly, with FFNN as a strong second.

## Complexity Robustness

How much each model's F_beta drops from low-complexity to high-complexity sessions:

| Model | Low (<=10) | High (>20) | Drop |
|-------|-----------|-----------|------|
| **CNN** | 0.746 | 0.690 | **-0.056 (most robust)** |
| FFNN | 0.705 | 0.663 | -0.042 |
| XGBoost | 0.731 | 0.617 | -0.114 |
| BiLSTM | 0.738 | 0.564 | -0.174 (steepest drop) |

CNN and FFNN handle sessions with many intervals much better than XGBoost and BiLSTM. This is an important practical consideration, since complex sessions with many intervals are precisely the ones where automated detection is most valuable.

## Practical Recommendation

- **Single-model deployment**: Use **CNN** -- it has the highest average performance, best robustness to complexity, lowest error rates, and best biking performance.
- **Sport-specific deployment**: Use **XGBoost for simple rowing sessions** (its sweet spot: low complexity + rowing) and **CNN for everything else** (biking, complex sessions, or when the session type is unknown).
- **Ensemble / User-in-the-Loop**: The top 4 models (CNN, FFNN, XGBoost, BiLSTM) agree on many predictions but disagree on edge cases. Using inter-model agreement as a confidence signal can further improve detection quality by auto-accepting high-agreement predictions and flagging disagreements for user review.

# User-in-the-Loop Interface Design

## Overview
A hybrid confidence-based approach that leverages inter-model agreement to reduce user burden while maintaining full control.

---

## Phase 1: Auto-Accept High-Confidence Predictions ✅

**Show automatically (pre-filled):**
- Intervals where **both models agree** (within 3 seconds)
- Display as **solid green markers**
- Label: "High confidence - both models agree"

**Visual cue:** These appear as **locked/confirmed** intervals that user can still edit if needed

**Statistics to show:**
- "12 out of 15 intervals auto-detected with high confidence"
- "Agreement rate: 80%"

---

## Phase 2: Show Disagreements as "Suggestions" 💡

**For intervals where models disagree:**

Display **shadow markers** (semi-transparent, not confirmed):
- 🔵 **Blue shadow X** = XGBoost suggests an interval here
- 🟠 **Orange shadow circle** = FFNN suggests an interval here
- 🟣 **Purple shadow** = Both suggest something, but >3 sec apart

**User actions:**
1. **Click to accept** a shadow → Converts to solid green
2. **Drag to adjust** timing if model is close but slightly off
3. **Ignore** if it's a false positive
4. **Add manually** if both models missed something

**Smart labeling:**
```
"XGBoost suggests interval (confidence: rowing patterns)"
"FFNN suggests interval (confidence: high-complexity session)"
```

---

## Phase 3: Contextual Decision Rules 🧠

When there are **too few agreements** (e.g., <50% of expected intervals), trigger different modes:

### Rule 1: Low Agreement on Rowing Session
```
Agreement rate: 40% (4/10 intervals)
Sport: Rowing

→ Default to XGBoost shadows (historically 80% accurate on rowing)
→ Show FFNN shadows as "alternative suggestions" (lighter/smaller)
→ Message: "XGBoost typically performs better on rowing sessions"
```

### Rule 2: Low Agreement on High-Complexity Biking
```
Agreement rate: 30% (12/40 intervals)
Sport: Biking, Complexity: High (40 intervals)

→ Show BOTH as equal shadows (no preference)
→ Message: "Complex session - both models show uncertainty.
           Manual review recommended for disagreements."
→ Enable "quick accept" mode: click once = accept nearest suggestion
```

---

## Interaction Modes

### Mode A: "Quick Review" (Default)
- Auto-accepted intervals: ✅ Locked green
- Shadow suggestions: Click to accept
- Estimated time: 30 seconds

### Mode C: "Manual Override"
- Hide all shadows
- User marks everything manually
- Use for: Unusual sessions, equipment issues, data quality problems

---

## Visual Hierarchy (Priority Order)

### Tier 1: HIGH CONFIDENCE
✅ **Solid green** = Both models agree (within 3 sec)
- Large, prominent markers
- Auto-accepted but editable
- ~60-80% of intervals (based on data)

### Tier 2: MEDIUM CONFIDENCE
💡 **Bright shadows** = One model suggests (the "preferred" one based on sport/complexity)
- XGBoost blue X for rowing
- FFNN orange circle for biking
- Standard size, easy to click

### Tier 3: LOW CONFIDENCE
🤔 **Faint shadows** = Alternative model suggests
- Smaller, more transparent
- Secondary option
- User can toggle visibility

### Tier 4: USER ADDED
➕ **Yellow marker** = User manually placed
- Distinguishable from auto-detected
- Helps identify model limitations

---

## Smart Features

### 1. Progressive Disclosure
```
Initial view: Only high-confidence + top suggestions
↓
Click "Show more suggestions" → Reveal lower-confidence shadows
↓
Click "Show all" → Even the weakest suggestions
```

### 2. Batch Operations
```
"Accept all XGBoost suggestions" (for rowing sessions)
"Accept all FFNN suggestions" (for biking sessions)
"Accept all agreements only"
```

### 3. Keyboard Shortcuts
```
Spacebar: Accept nearest shadow
X: Delete nearest marker
A: Accept all visible suggestions
M: Add manual interval
```

---

## Decision Tree for Shadow Display
```
START
├─ Both agree (≤3 sec apart)?
│  └─ YES → ✅ Auto-accept (green)
│
├─ NO → Check sport + complexity
│
├─ Rowing + Low/Med complexity?
│  └─ Show XGBoost as primary shadow (blue)
│  └─ FFNN as secondary (faint orange)
│
├─ Biking + Any complexity?
│  └─ Show FFNN as primary shadow (orange)
│  └─ XGBoost as secondary (faint blue)
│
├─ High complexity (>20) + Biking?
│  └─ Show BOTH equally (no hierarchy)
│
└─ Agreement rate <40%?
   └─ Show both + warning message
   └─ Suggest "Manual Override" mode
```

---

## Why This Approach Works

### 1. Reduces Cognitive Load
- User sees ~60-80% pre-filled (agreements)
- Only needs to review 20-40% (disagreements)
- Clear visual hierarchy guides attention

### 2. Maintains Control
- User can override everything
- Can see what models suggest without forcing acceptance
- Progressive detail levels

### 3. Builds Trust Over Time
- Initial skepticism: Review everything
- After 5 sessions: "Agreements are always right"
- After 10 sessions: "I can trust XGBoost on rowing"

### 4. Handles Edge Cases
- Low agreement? Fall back to manual with smart suggestions
- Unusual session? Mode C: full manual

### 5. Measurable Improvement
Track metrics:
- Time to complete review (should decrease)
- Override rate (should decrease)
- User satisfaction (should increase)

---

## Thesis Summary

> "Rather than selecting a single 'best' model, we developed a **confidence-tiered user interface** that presents interval predictions based on inter-model agreement. High-confidence predictions (model agreement) are auto-accepted, reducing user burden by ~70%, while disagreements are presented as contextual suggestions informed by sport-specific performance patterns. This hybrid approach leverages the complementary strengths of both models while maintaining full user control for edge cases."

**Key principle:** Use model agreement as the primary confidence signal, and disagreements as valuable information about uncertainty.# User-in-the-Loop Interface Design

## Overview
A hybrid confidence-based approach that leverages inter-model agreement to reduce user burden while maintaining full control.

---

## Phase 1: Auto-Accept High-Confidence Predictions ✅

**Show automatically (pre-filled):**
- Intervals where **both models agree** (within 3 seconds)
- Display as **solid green markers**
- Label: "High confidence - both models agree"

**Visual cue:** These appear as **locked/confirmed** intervals that user can still edit if needed

**Statistics to show:**
- "12 out of 15 intervals auto-detected with high confidence"
- "Agreement rate: 80%"

---

## Phase 2: Show Disagreements as "Suggestions" 💡

**For intervals where models disagree:**

Display **shadow markers** (semi-transparent, not confirmed):
- 🔵 **Blue shadow X** = XGBoost suggests an interval here
- 🟠 **Orange shadow circle** = FFNN suggests an interval here
- 🟣 **Purple shadow** = Both suggest something, but >3 sec apart

**User actions:**
1. **Click to accept** a shadow → Converts to solid green
2. **Drag to adjust** timing if model is close but slightly off
3. **Ignore** if it's a false positive
4. **Add manually** if both models missed something

**Smart labeling:**
```
"XGBoost suggests interval (confidence: rowing patterns)"
"FFNN suggests interval (confidence: high-complexity session)"
```

---

## Phase 3: Contextual Decision Rules 🧠

When there are **too few agreements** (e.g., <50% of expected intervals), trigger different modes:

### Rule 1: Low Agreement on Rowing Session
```
Agreement rate: 40% (4/10 intervals)
Sport: Rowing

→ Default to XGBoost shadows (historically 80% accurate on rowing)
→ Show FFNN shadows as "alternative suggestions" (lighter/smaller)
→ Message: "XGBoost typically performs better on rowing sessions"
```

### Rule 2: Low Agreement on High-Complexity Biking
```
Agreement rate: 30% (12/40 intervals)
Sport: Biking, Complexity: High (40 intervals)

→ Show BOTH as equal shadows (no preference)
→ Message: "Complex session - both models show uncertainty.
           Manual review recommended for disagreements."
→ Enable "quick accept" mode: click once = accept nearest suggestion
```

---

## Interaction Modes

### Mode A: "Quick Review" (Default)
- Auto-accepted intervals: ✅ Locked green
- Shadow suggestions: Click to accept
- Estimated time: 30 seconds

### Mode C: "Manual Override"
- Hide all shadows
- User marks everything manually
- Use for: Unusual sessions, equipment issues, data quality problems

---

## Visual Hierarchy (Priority Order)

### Tier 1: HIGH CONFIDENCE
✅ **Solid green** = Both models agree (within 3 sec)
- Large, prominent markers
- Auto-accepted but editable
- ~60-80% of intervals (based on data)

### Tier 2: MEDIUM CONFIDENCE
💡 **Bright shadows** = One model suggests (the "preferred" one based on sport/complexity)
- XGBoost blue X for rowing
- FFNN orange circle for biking
- Standard size, easy to click

### Tier 3: LOW CONFIDENCE
🤔 **Faint shadows** = Alternative model suggests
- Smaller, more transparent
- Secondary option
- User can toggle visibility

### Tier 4: USER ADDED
➕ **Yellow marker** = User manually placed
- Distinguishable from auto-detected
- Helps identify model limitations

---

## Smart Features

### 1. Progressive Disclosure
```
Initial view: Only high-confidence + top suggestions
↓
Click "Show more suggestions" → Reveal lower-confidence shadows
↓
Click "Show all" → Even the weakest suggestions
```

### 2. Batch Operations
```
"Accept all XGBoost suggestions" (for rowing sessions)
"Accept all FFNN suggestions" (for biking sessions)
"Accept all agreements only"
```

### 3. Keyboard Shortcuts
```
Spacebar: Accept nearest shadow
X: Delete nearest marker
A: Accept all visible suggestions
M: Add manual interval
```

---

## Decision Tree for Shadow Display
```
START
├─ Both agree (≤3 sec apart)?
│  └─ YES → ✅ Auto-accept (green)
│
├─ NO → Check sport + complexity
│
├─ Rowing + Low/Med complexity?
│  └─ Show XGBoost as primary shadow (blue)
│  └─ FFNN as secondary (faint orange)
│
├─ Biking + Any complexity?
│  └─ Show FFNN as primary shadow (orange)
│  └─ XGBoost as secondary (faint blue)
│
├─ High complexity (>20) + Biking?
│  └─ Show BOTH equally (no hierarchy)
│
└─ Agreement rate <40%?
   └─ Show both + warning message
   └─ Suggest "Manual Override" mode
```

---

## Why This Approach Works

### 1. Reduces Cognitive Load
- User sees ~60-80% pre-filled (agreements)
- Only needs to review 20-40% (disagreements)
- Clear visual hierarchy guides attention

### 2. Maintains Control
- User can override everything
- Can see what models suggest without forcing acceptance
- Progressive detail levels

### 3. Builds Trust Over Time
- Initial skepticism: Review everything
- After 5 sessions: "Agreements are always right"
- After 10 sessions: "I can trust XGBoost on rowing"

### 4. Handles Edge Cases
- Low agreement? Fall back to manual with smart suggestions
- Unusual session? Mode C: full manual

### 5. Measurable Improvement
Track metrics:
- Time to complete review (should decrease)
- Override rate (should decrease)
- User satisfaction (should increase)

---

## Thesis Summary

> "Rather than selecting a single 'best' model, we developed a **confidence-tiered user interface** that presents interval predictions based on inter-model agreement. High-confidence predictions (model agreement) are auto-accepted, reducing user burden by ~70%, while disagreements are presented as contextual suggestions informed by sport-specific performance patterns. This hybrid approach leverages the complementary strengths of both models while maintaining full user control for edge cases."

**Key principle:** Use model agreement as the primary confidence signal, and disagreements as valuable information about uncertainty.